In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [24]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

In [4]:
loader=PyPDFLoader("../data/data_science_syllabus.pdf")
docs=loader.load()
len(docs)

10

In [7]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_data=splitter.split_documents(docs)
len(splitted_data)

11

In [8]:
embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [15]:
vector_store=Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [34]:
query="Machine learnigs and Data Science Content"
data=vector_store.similarity_search(query=query)

In [17]:
len(data)

4

In [18]:
data[0]

Document(id='45e9bc5f-a503-4a7b-ae22-e58c8f770da8', metadata={'source': '../data/data_science_syllabus.pdf', 'page_label': '10', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/137.0.0.0 Safari/537.36', 'creationdate': '2025-12-13T08:07:26+00:00', 'total_pages': 10, 'producer': 'Skia/PDF m137', 'page': 9, 'moddate': '2025-12-13T08:07:26+00:00', 'title': '📘 1-Year Roadmap: Data Analytics, Data Science & GenAI'}, page_content='📘  1\ue088 Y e ar R o admap: Dat a Anal y tics, Dat a Science & GenAI\n10')

In [38]:
context=""
for doc in data:
    context += doc.page_content + "\n"
print(context)

📘  1 Y e ar R o admap: Dat a Anal y tics, Dat a Science & GenAI
10
Data Analytics Dashboard (end-to-end BI project)
ML Web App (e.g., Predict & display output with Streamlit)
LLM Chatbot with LangChain  Vector DB
YouTube Transcript Analyzer using Whisper  GPT
📘  1 Y e ar R o admap: Dat a Anal y tics, Dat a Science & GenAI
9
Customer Segmentation
E-commerce Product Clustering
🎤  Mock Interview: Unsupervised Learning  Features
✅  Module 8: Time Series + Intro to Deep Learning (3 
weeks)
Duration: Month 8
Topics:
Time Series components, lag, rolling stats
Forecasting: ARIMA, Prophet
Intro to Neural Networks
Perceptron, basic Keras model
Tools:
Statsmodels, Prophet, Keras, TensorFlow
Mini Project:
Stock Price or Weather Forecasting
🎤  Mock Interview: Forecasting  Neural Net Basics
✅  Module 8.5: Big Data with PySpark (2 weeks)
Duration: Between Month 89
Topics:
What is Big Data?
PySpark architecture & setup
DataFrames, RDDs, Transformations, Actions
PySpark SQL  basic MLlib
Databri

In [35]:
llm=ChatGroq(model="llama-3.1-8b-instant")

In [39]:
res=llm.invoke("""Can you provide me the answer based on 
               provided context for my question,context: {context} and question: {query}""")  

In [40]:
print(res.content)

It seems that you want to provide a context and a question, and I'll do my best to answer it.

Please go ahead and provide the context and the question, and I'll do my best to assist you. 

For example, you can put the context in the following format: 

```
{context}
```

And the question in the following format:

```
{query}
```

Once you do that, I'll be happy to answer your question.


CHAIN: CONTEXT_GENERATE | PROMPT | LLM | strparser

In [27]:
def get_context(query:str):
    data=vector_store.similarity_search(query=query)
    context=""
    for doc in data:
        context += doc.page_content + "\n"
    return{
        "context":context,
        "question":query
        
    }
    

In [42]:
prompt=PromptTemplate.from_template("""
    You are a helpful assistant and provide answer based on the context for user question and if you don't know the answer, then you can say "I don't know the answer." 
    context:{context}
    question:{question}                               
""")

In [43]:
rag_chain=get_context | prompt | llm

In [44]:
res=rag_chain.invoke("What is the duration of my course ?")

In [45]:
print(res.content)

Based on the provided information, the duration of your course is 5 months, with each month dedicated to a specific module:

- Month 1: Module 1 - Python Programming for Data & AI (4 weeks)
- Month 2: Module 2 - Data Handling with Pandas & Numpy (4 weeks)
- Month 3: Module 3 - SQL & Excel for Analysts (3 weeks)
- Month 4: Module 4 - Data Visualization & Storytelling (3 weeks)
- Month 5: Module 5 - Statistics & Probability for Data Science (4 weeks)

However, there is also a 1-2 day duration for Module 0 - Orientation & Core Concepts at the beginning of the course. 

So, in total, the course duration is approximately 5 months and 2 days.


In [46]:
res=rag_chain.invoke("What is the content of JAVA DSA Course ?")

In [47]:
print(res.content)

I don't know the answer. The provided context does not mention JAVA DSA Course. It seems to be related to a Cloud Data Engineer and AI Product Analyst course, covering various tools and technologies such as Python, SQL, Data Science, GenAI, and Cloud deployment. If you have any further questions related to this course, I'll be happy to help.
